# Agente GenAI de E-Commerce
## Notebook de Demonstração

---

Este notebook demonstra as capacidades do agente de análise de dados construído com **LangGraph** e **Gemini 2.5 Flash-Lite**.

O agente recebe perguntas em português, converte para SQL, executa contra um banco SQLite e retorna análise estruturada com geração automática de gráficos.

**Pré-requisitos**
- Arquivo `banco.db` posicionado na raiz do projeto
- Variável `GOOGLE_API_KEY` configurada no arquivo `.env`
- Dependências instaladas via `pip install -r requirements.txt`

---

**Seções**

1. Configuração do Ambiente
2. Análise de Vendas e Receita
3. Análise de Logística e Entregas
4. Análise de Satisfação e Avaliações
5. Conversa Multi-turno com Memória de Sessão
6. Validação dos Guardrails de Segurança
7. Acesso Direto ao Banco e Exportação
8. Dashboard Automático de Insights

---
## 1. Configuração do Ambiente

In [ ]:
import sys
import os
import json
import pandas as pd
import plotly.graph_objects as go

RAIZ_PROJETO = os.path.abspath('..')
if RAIZ_PROJETO not in sys.path:
    sys.path.insert(0, RAIZ_PROJETO)

from dotenv import load_dotenv
load_dotenv(os.path.join(RAIZ_PROJETO, '.env'))

from app.agent.agent import chat, clear_session
from app.agent.guardrails import validate_sql, GuardrailViolation
from app.agent.insights import run_insights
from app.charts.generator import generate_chart, dataframe_from_result
from app.database.connection import query_to_dataframe

ID_SESSAO = 'notebook_demo'
clear_session(ID_SESSAO)

print('Ambiente configurado.')
print(f'Modelo: {os.getenv("GEMINI_MODEL", "gemini-2.5-flash-lite")}')
print(f'Banco de dados: {os.getenv("DB_PATH", "banco.db")}')

---
## 2. Análise de Vendas e Receita

Demonstra o pipeline Text-to-SQL com uma consulta de ranking. O agente gera o SQL, executa e retorna análise em linguagem natural.

In [ ]:
resultado = chat(ID_SESSAO, 'Quais sao os 10 produtos mais vendidos? Mostre a categoria e a quantidade total vendida.')

print('--- Resposta do Agente ---')
print(resultado['response'])

print('\n--- SQL Gerado ---')
for sql in resultado['sql_queries']:
    print(sql)

In [ ]:
# Renderizar o grafico retornado pelo agente
if resultado['query_results']:
    df = dataframe_from_result(resultado['query_results'][-1])
    grafico = generate_chart(df, title='Top 10 Produtos Mais Vendidos')
    if grafico:
        fig = go.Figure(grafico)
        fig.show()
    display(df)

In [ ]:
resultado_receita = chat(ID_SESSAO, 'Qual a receita total por categoria de produto? Considere apenas pedidos entregues.')

print('--- Resposta do Agente ---')
print(resultado_receita['response'])

if resultado_receita['query_results']:
    df = dataframe_from_result(resultado_receita['query_results'][-1])
    grafico = generate_chart(df, title='Receita Total por Categoria de Produto')
    if grafico:
        fig = go.Figure(grafico)
        fig.show()

---
## 3. Análise de Logística e Entregas

Consultas com filtros de data e agrupamento geográfico.

In [ ]:
resultado_entrega = chat(ID_SESSAO, 'Qual o percentual de pedidos entregues no prazo por estado? Ordene do pior para o melhor.')

print('--- Resposta do Agente ---')
print(resultado_entrega['response'])

print('\n--- SQL Gerado ---')
for sql in resultado_entrega['sql_queries']:
    print(sql)

if resultado_entrega['query_results']:
    df = dataframe_from_result(resultado_entrega['query_results'][-1])
    grafico = generate_chart(df, title='Percentual de Entregas no Prazo por Estado')
    if grafico:
        fig = go.Figure(grafico)
        fig.show()

In [ ]:
resultado_status = chat(ID_SESSAO, 'Quantos pedidos existem por status?')

print('--- Resposta do Agente ---')
print(resultado_status['response'])

if resultado_status['query_results']:
    df = dataframe_from_result(resultado_status['query_results'][-1])
    grafico = generate_chart(df, title='Volume de Pedidos por Status')
    if grafico:
        fig = go.Figure(grafico)
        fig.show()

---
## 4. Análise de Satisfação e Avaliações

Agrupamento de notas de avaliacao por vendedor e por categoria.

In [ ]:
resultado_avaliacao = chat(ID_SESSAO, 'Qual a media de avaliacao por vendedor? Mostre o top 10 com pelo menos 20 avaliacoes.')

print('--- Resposta do Agente ---')
print(resultado_avaliacao['response'])

if resultado_avaliacao['query_results']:
    df = dataframe_from_result(resultado_avaliacao['query_results'][-1])
    display(df)

In [ ]:
resultado_negativo = chat(ID_SESSAO, 'Quais categorias tem maior taxa de avaliacao negativa (nota 1 ou 2)?')

print('--- Resposta do Agente ---')
print(resultado_negativo['response'])

if resultado_negativo['query_results']:
    df = dataframe_from_result(resultado_negativo['query_results'][-1])
    grafico = generate_chart(df, title='Categorias com Maior Taxa de Avaliacao Negativa')
    if grafico:
        fig = go.Figure(grafico)
        fig.show()

---
## 5. Conversa Multi-turno com Memória de Sessão

O agente preserva o contexto entre mensagens da mesma sessão. A pergunta de acompanhamento abaixo nao repete nenhuma informacao anterior — o agente infere o contexto pelo historico.

In [ ]:
#estabelece o contexto
r1 = chat(ID_SESSAO, 'Quais estados tem maior volume de pedidos?')
print('Turno 1')
print(r1['response'])

In [ ]:
#pergunta de acompanhamento sem repetir o tema
#O agente usa a memoria da sessao para entender a que 'esses estados' se refere
r2 = chat(ID_SESSAO, 'E qual o ticket medio nesses estados?')
print('Turno 2 (o agente lembra o contexto do turno anterior)')
print(r2['response'])

In [ ]:
#aprofunda a analise
r3 = chat(ID_SESSAO, 'Qual deles tem maior atraso medio de entrega?')
print('Turno 3')
print(r3['response'])

---
## 6. Validação dos Guardrails de Segurança

A camada de guardrails valida toda string SQL antes da execucao. Apenas instrucoes `SELECT` e CTEs (`WITH ... SELECT`) sao permitidas. Todas as operacoes de escrita e modificacao de schema sao bloqueadas incondicionalmente.

In [ ]:
casos_de_teste = [
    ('SELECT COUNT(*) FROM fat_pedidos',                                    'PERMITIDO'),
    ('WITH top AS (SELECT id_pedido FROM fat_pedidos) SELECT * FROM top',   'PERMITIDO'),
    ('DELETE FROM fat_pedidos',                                             'BLOQUEADO'),
    ('DROP TABLE dim_produtos',                                             'BLOQUEADO'),
    ('INSERT INTO fat_pedidos VALUES (1)',                                  'BLOQUEADO'),
    ('UPDATE dim_produtos SET nome_produto = "hack"',                      'BLOQUEADO'),
    ('SELECT * FROM fat_pedidos; DROP TABLE fat_pedidos',                  'BLOQUEADO'),
    ('PRAGMA table_info(fat_pedidos)',                                      'BLOQUEADO'),
]

print(f'{"Query":<60} {"Esperado":<12} {"Resultado"}')
print('-' * 92)

todos_passaram = True
for query, esperado in casos_de_teste:
    try:
        validate_sql(query)
        resultado = 'PERMITIDO'
    except GuardrailViolation:
        resultado = 'BLOQUEADO'

    passou = resultado == esperado
    if not passou:
        todos_passaram = False
    status = 'OK' if passou else 'FALHOU'
    print(f'{query[:58]:<60} {esperado:<12} {status}')

print('-' * 92)
print(f'Todos os testes passaram: {todos_passaram}')

---
## 7. Acesso Direto ao Banco e Exportação

A camada de banco pode ser utilizada independentemente do agente — util para scripts de analise reproduzivel ou geracao de relatorios agendados.

In [ ]:
df_estados = query_to_dataframe("""
    SELECT
        dc.estado,
        COUNT(DISTINCT fp.id_pedido)           AS total_pedidos,
        ROUND(AVG(ft.valor_total_pago_brl), 2) AS ticket_medio_brl,
        ROUND(AVG(fp.tempo_entrega_dias), 1)   AS tempo_medio_entrega_dias
    FROM fat_pedidos fp
    JOIN dim_consumidores dc ON fp.id_consumidor = dc.id_consumidor
    JOIN fat_pedido_total  ft ON fp.id_pedido     = ft.id_pedido
    WHERE fp.status IN ('entregue', 'delivered')
    GROUP BY dc.estado
    ORDER BY total_pedidos DESC
""")

print(f'Linhas retornadas: {len(df_estados)}')
display(df_estados.head(10))

In [ ]:
#exportacao para CSV e Excel
diretorio_saida = os.path.join(RAIZ_PROJETO, 'notebooks')

caminho_csv   = os.path.join(diretorio_saida, 'analise_estados.csv')
caminho_excel = os.path.join(diretorio_saida, 'analise_estados.xlsx')

df_estados.to_csv(caminho_csv, index=False, encoding='utf-8')
df_estados.to_excel(caminho_excel, index=False, engine='openpyxl')

print(f'CSV exportado:   {caminho_csv}')
print(f'Excel exportado: {caminho_excel}')

In [ ]:
#renderizacao direta do dataframe como grafico
grafico = generate_chart(
    df_estados[['estado', 'ticket_medio_brl']].head(10),
    title='Ticket Medio por Estado (Top 10)'
)
if grafico:
    fig = go.Figure(grafico)
    fig.show()

---
## 8. Dashboard Automático de Insights

O modulo de insights executa um conjunto de analises pre-definidas sem necessidade de pergunta do usuario. Ele detecta os nomes reais das colunas do banco em tempo de execucao e constroi o SQL dinamicamente, tornando-o resiliente a variacoes de schema.

In [ ]:
analises = run_insights()

for item in analises:
    print(f"\n{'=' * 60}")
    print(f"  {item['title']}")
    print(f"  {item['description']}")
    print(f"{'=' * 60}")

    if item['error']:
        print(f"  Erro: {item['error']}")
        continue

    print(f"  Linhas retornadas: {item['rows']}")

    if item['data']:
        df = pd.DataFrame(item['data'])
        display(df.head(5))

    if item['chart']:
        fig = go.Figure(item['chart'])
        fig.show()

---

## Resumo

| Secao | Funcionalidade demonstrada |
|---|---|
| 2 | Pipeline Text-to-SQL, geracao automatica de graficos |
| 3 | Filtros por data, agrupamento geografico |
| 4 | Agregacao de notas de avaliacao por vendedor e categoria |
| 5 | Memoria de sessao multi-turno (3 turnos encadeados) |
| 6 | Camada de guardrails — 8 casos de teste, 100% de aprovacao |
| 7 | Acesso direto ao banco, exportacao CSV e Excel |
| 8 | Dashboard automatico com deteccao dinamica de schema |

O servidor FastAPI (`python run.py`) expoe todas essas capacidades via API REST documentada em `/docs`, alem da interface de chat em `/`.